# Setup

In [1]:
import warnings
warnings.filterwarnings('ignore', message='BigQuery Storage module not found')

import time
import numpy as np
import random
import requests
from google.cloud import bigquery

# ── Endpoint ────────────────────────────────────────────────────────────
ENDPOINT_URL = 'https://ternopil-tern-nbo-v1-serving-3dmqemfmxq-ez.a.run.app'
MODEL_NAME   = 'recommender'
PREDICT_URL  = f'{ENDPOINT_URL}/v1/models/{MODEL_NAME}:predict'
BATCH_SIZE   = 64           # native TF Serving max_batch_size limit

GCP_PROJECT = 'b2b-recs'
BQ_TRAIN = f'{GCP_PROJECT}.raw_data.ternopil_train_v4'
BQ_TEST  = f'{GCP_PROJECT}.raw_data.ternopil_test_v4'
client = bigquery.Client(project=GCP_PROJECT)

HISTORY_LEN = 50

def pad_history(hist, max_len=HISTORY_LEN):
    """Pad/truncate purchase history to fixed length, 0-padded.
    Converts numpy int64 → Python int for JSON serialization."""
    arr = [int(x) for x in hist] if hist is not None else []
    arr = arr[:max_len]
    return arr + [0] * (max_len - len(arr))

def predict(instances):
    """Call endpoint with automatic batching.
    Returns (product_ids, scores) as numpy arrays."""
    all_pids, all_scores = [], []
    for i in range(0, len(instances), BATCH_SIZE):
        batch = instances[i:i + BATCH_SIZE]
        resp = requests.post(PREDICT_URL, json={'instances': batch}, timeout=120)
        resp.raise_for_status()
        for p in resp.json()['predictions']:
            all_pids.append(p['product_ids'])
            all_scores.append(p['scores'])
    return np.array(all_pids, dtype=np.int64), np.array(all_scores, dtype=np.float32)

# ── Verify endpoint ─────────────────────────────────────────────────────
resp = requests.get(f'{ENDPOINT_URL}/v1/models/{MODEL_NAME}', timeout=30)
print(f'Endpoint: {"OK" if resp.ok else "UNREACHABLE " + str(resp.status_code)}')


Endpoint: OK


# Random product

In [10]:
# Re-run to get a different customer/product
# Pick a random interaction from the TEST set (held-out last day)
from IPython.display import display, HTML

row = client.query(f"""
SELECT
    CAST(customer_id AS INT64) AS customer_id,
    cust_value,
    UNIX_SECONDS(date) AS date_unix,
    mge_cat_desc AS sub_category_v2,
    CAST(product_id AS INT64) AS product_id,
    art_name,
    brand_name AS brand,
    stratbuy_domain_desc AS category,
    mge_main_cat_desc AS sub_category_v1,
    days_since_last_purchase,
    cust_order_days_60d,
    cust_unique_products_60d,
    purchase_history
FROM `{BQ_TEST}`
WHERE cust_value > 0
ORDER BY RAND() LIMIT 1
""").to_dataframe().iloc[0]

cust_id = int(row['customer_id'])
cust_value = float(row['cust_value'])
date_unix = int(row['date_unix'])
input_subcat_v2 = row['sub_category_v2']
input_subcat_v1 = row['sub_category_v1']
input_cat = row['category']
cust_last_purchase = int(row['days_since_last_purchase'] or 0)
cust_visits = int(row['cust_order_days_60d'] or 0)
cust_bought_SKU = int(row['cust_unique_products_60d'] or 0)
actual_product = int(row['product_id'])
history_padded = pad_history(row['purchase_history'])

display(HTML(f"""
<table style="border-collapse:collapse; font-size:14px;">
  <tr><th style="text-align:left; padding:4px 12px; border:1px solid #ccc;">customer_id</th>
      <td style="padding:4px 12px; border:1px solid #ccc;">{cust_id}</td></tr>
  <tr><th style="text-align:left; padding:4px 12px; border:1px solid #ccc;">date</th>
      <td style="padding:4px 12px; border:1px solid #ccc;">{date_unix}</td></tr>
  <tr><th style="text-align:left; padding:4px 12px; border:1px solid #ccc;">cust_value</th>
      <td style="padding:4px 12px; border:1px solid #ccc;">{cust_value:.0f}</td></tr>
  <tr><th style="text-align:left; padding:4px 12px; border:1px solid #ccc;">cust_last_purchase</th>
      <td style="padding:4px 12px; border:1px solid #ccc;">{cust_last_purchase}</td></tr>
  <tr><th style="text-align:left; padding:4px 12px; border:1px solid #ccc;">cust_visits</th>
      <td style="padding:4px 12px; border:1px solid #ccc;">{cust_visits}</td></tr>
  <tr><th style="text-align:left; padding:4px 12px; border:1px solid #ccc;">cust_bought_SKU</th>
      <td style="padding:4px 12px; border:1px solid #ccc;">{cust_bought_SKU}</td></tr>
  <tr><th style="text-align:left; padding:4px 12px; border:1px solid #ccc;">history</th>
      <td style="padding:4px 12px; border:1px solid #ccc;">{sum(1 for x in history_padded if x != 0)} products</td></tr>
  <tr><th style="text-align:left; padding:4px 12px; border:1px solid #ccc;">product</th>
      <td style="padding:4px 12px; border:1px solid #ccc;">{row['art_name']}</td></tr>
  <tr><th style="text-align:left; padding:4px 12px; border:1px solid #ccc;">brand</th>
      <td style="padding:4px 12px; border:1px solid #ccc;">{row['brand']}</td></tr>
  <tr><th style="text-align:left; padding:4px 12px; border:1px solid #ccc;">category</th>
      <td style="padding:4px 12px; border:1px solid #ccc;">{input_cat}</td></tr>
  <tr><th style="text-align:left; padding:4px 12px; border:1px solid #ccc;">sub_category (v1)</th>
      <td style="padding:4px 12px; border:1px solid #ccc;">{input_subcat_v1}</td></tr>
  <tr><th style="text-align:left; padding:4px 12px; border:1px solid #ccc;">sub_category (v2)</th>
      <td style="padding:4px 12px; border:1px solid #ccc;">{input_subcat_v2}</td></tr>
</table>
"""))


customer_id,640091016801
date,1717113600
cust_value,8173
cust_last_purchase,1
cust_visits,18
cust_bought_SKU,32
history,20 products
product,LA BARRICA ХАМОН РЕЗЕРВА 90Г
brand,LA BARRICA
category,PROCESSED MEATS
sub_category (v1),СИРОКОПЧЕНА ШИНКА/М'ЯСО


# Top-20 recs

In [11]:
# Inference via endpoint — payload: customer_id, date, cust_value, sub_cat_v2, cs_last_purch, cs_orders, cs_unique_skus, history
from IPython.display import display, HTML

# Products to suppress from recommendations (noisy/dominant items)
SUPPRESS_NAMES = {
    'ARO \u0426\u0423\u041a\u041e\u0420 50\u041a\u0413',
    'MPRO \u041f\u0410\u041a\u0415\u0422 \u0412\u0415\u041b 51\u041c\u041a\u041c 45*74',
}

instance = {
    'customer_id':   cust_id,
    'date':          date_unix,
    'cust_value':    cust_value,
    'sub_cat_v2':    input_subcat_v2,
    'cs_last_purch': cust_last_purchase,
    'cs_orders':     cust_visits,
    'cs_unique_skus': cust_bought_SKU,
    'history':       history_padded,
}
rec_ids_arr, rec_scores_arr = predict([instance])
rec_ids = rec_ids_arr[0]
rec_scores = rec_scores_arr[0]

# Enrich from training data (broader product coverage) — include art_name
ids_str = ', '.join(str(int(p)) for p in rec_ids)
meta = client.query(f"""
SELECT DISTINCT
    CAST(product_id AS INT64) AS product_id,
    art_name,
    brand_name AS brand,
    stratbuy_domain_desc AS category,
    mge_main_cat_desc AS sub_category_v1,
    mge_cat_desc AS sub_category_v2
FROM `{BQ_TRAIN}`
WHERE product_id IN ({ids_str})
""").to_dataframe().set_index('product_id').to_dict('index')

# Build curated top-20: sub_category_v2 → sub_category_v1 → category → rest
TOP_N = 20
seen = {actual_product}  # exclude the input product from recommendations
curated = []  # list of (product_id, score)

# Pass 1: same sub_category_v2 (finest granularity, buyer context feature)
for i in range(len(rec_ids)):
    pid = int(rec_ids[i])
    if pid in seen:
        continue
    m = meta.get(pid, {})
    if m.get('art_name') in SUPPRESS_NAMES:
        continue
    if m.get('sub_category_v2') == input_subcat_v2:
        curated.append((pid, float(rec_scores[i])))
        seen.add(pid)

# Pass 2: same sub_category_v1 (broader sub-category)
for i in range(len(rec_ids)):
    if len(curated) >= TOP_N:
        break
    pid = int(rec_ids[i])
    if pid in seen:
        continue
    m = meta.get(pid, {})
    if m.get('art_name') in SUPPRESS_NAMES:
        continue
    if m.get('sub_category_v1') == input_subcat_v1:
        curated.append((pid, float(rec_scores[i])))
        seen.add(pid)

# Pass 3: same category
for i in range(len(rec_ids)):
    if len(curated) >= TOP_N:
        break
    pid = int(rec_ids[i])
    if pid in seen:
        continue
    m = meta.get(pid, {})
    if m.get('art_name') in SUPPRESS_NAMES:
        continue
    if m.get('category') == input_cat:
        curated.append((pid, float(rec_scores[i])))
        seen.add(pid)

# Pass 4: remaining by score
for i in range(len(rec_ids)):
    if len(curated) >= TOP_N:
        break
    pid = int(rec_ids[i])
    if pid in seen:
        continue
    m = meta.get(pid, {})
    if m.get('art_name') in SUPPRESS_NAMES:
        continue
    curated.append((pid, float(rec_scores[i])))
    seen.add(pid)

curated = curated[:TOP_N]

hdr = "".join(f'<th style=\"padding:4px 10px; border:1px solid #ccc; background:#eee;\">{c}</th>'
              for c in ['#', 'Product', 'Brand', 'Category', 'Sub_category (v1)', 'Sub_category (v2)', 'Product ID', 'Score'])

rows = []
for rank, (pid, score) in enumerate(curated, 1):
    m = meta.get(pid, {})
    cells = [
        f'{rank}',
        m.get('art_name', '?'),
        m.get('brand', '?'),
        m.get('category', '?'),
        m.get('sub_category_v1', '?'),
        m.get('sub_category_v2', '?'),
        str(pid),
        f'{score:.2f}',
    ]
    tds = "".join(f'<td style=\"padding:4px 10px; border:1px solid #ccc;\">{v}</td>' for v in cells)
    rows.append(f'<tr>{tds}</tr>')

display(HTML(f"""
<p style=\"font-size:13px; color:#666;\">Suppressed: {SUPPRESS_NAMES}</p>
<table style=\"border-collapse:collapse; font-size:13px;\">
  <tr>{hdr}</tr>
  {chr(10).join('  ' + r for r in rows)}
</table>
"""))


#,Product,Brand,Category,Sub_category (v1),Sub_category (v2),Product ID,Score
1,MC ХАМОН СЕРРАНО 300Г,METRO CHEF,PROCESSED MEATS,СИРОКОПЧЕНА ШИНКА/М'ЯСО,ІСПАНСЬКА СИРА ШИНКА,340974001001,451.39
2,GALAR ХАМОН СЕР.НАР. 500Г,GALAR,PROCESSED MEATS,СИРОКОПЧЕНА ШИНКА/М'ЯСО,ІСПАНСЬКА СИРА ШИНКА,327952001001,450.07
3,MC ХАМОН КУРАДО 300Г,METRO CHEF,PROCESSED MEATS,СИРОКОПЧЕНА ШИНКА/М'ЯСО,ІСПАНСЬКА СИРА ШИНКА,340973001001,439.23
4,MC ХАМОН МІНІ ПІДСТ.НІЖ 1КГ,METRO CHEF,PROCESSED MEATS,СИРОКОПЧЕНА ШИНКА/М'ЯСО,ІСПАНСЬКА СИРА ШИНКА,350782001001,438.05
5,MC ПАНЧЕТА 300Г,METRO CHEF,PROCESSED MEATS,СИРОКОПЧЕНА ШИНКА/М'ЯСО,ІТАЛІЙСЬКА СИРА ШИНКА,340969001001,416.01
6,MC ФУЕТ ЕКСТ. 170Г,METRO CHEF,PROCESSED MEATS,КОВБАСИ,СУШЕНА КОВБАСА,340972001001,388.89
7,RIOBA ЗЕРНА 100% АРАБІКА 1КГ,RIOBA,HOT BEVERAGES,КАВА,КАВА В ЗЕРНАХ,278070001001,447.90
8,RIOBA ЗЕРНА 55% АР 45% РОБ 1КГ,RIOBA,HOT BEVERAGES,КАВА,КАВА В ЗЕРНАХ,278065001001,443.27
9,ARO КОРМ Д/СОБ.СУХИЙ 10КГ,ARO,CANNED GOODS,КОРМ ДЛЯ ТВАРИН,СУХИЙ КОРМ ДЛЯ СОБАК,148134001001,442.15
10,RIOBA DOLCE ЗЕРНА 80% 20% 1КГ,RIOBA,HOT BEVERAGES,КАВА,КАВА В ЗЕРНАХ,223731001001,441.30
